In [ ]:
import itertools
from collections import defaultdict
from collections.abc import Mapping
from typing import NamedTuple

# from stereomolgraph.graphs.scrg import Change, Stereo
from stereomolgraph import AtomId, MolGraph
from stereomolgraph.algorithms.color_refine import color_refine_mg

# TODO:
# define priority based labels:
# - rearest color
# - finishes stereodescriptor
# - degree


In [ ]:
CanonNum = int

LabelClassSize = int
Degree = int
Label = int


class _Parameters(NamedTuple):
    """Parameters that are invariant across the search.

    :ivar nbrhds: The neighborhood of each atom in the graph.
    :ivar labels: A tuple of (label class size, degree, label) for each atom.
    """

    nbrhds: Mapping[AtomId, set[AtomId]] = defaultdict(set)
    labels: Mapping[AtomId, tuple[LabelClassSize, Degree, Label]] = {}

    # label_class_sizes: Mapping[Label, int] = {}
    # degrees: Mapping[AtomId, int] = {}

    # stereo
    # stereo_change
    # ...


class _CanonState(NamedTuple):
    best_labels: list = []
    best_mapping: dict[AtomId, CanonNum] = {}

    current_labels: list = []
    current_mapping: dict[AtomId, CanonNum] = {}

    frontier: set[AtomId] = set()

    atom_queue: set[AtomId] = set()
    backtrack_candidates_list: list[set[AtomId]] = []


In [ ]:
def initialize(g: MolGraph) -> tuple[_Parameters, _CanonState]:
    """Prepare immutable inputs and initial state for non-stereo canonical enumeration."""
    nbrhd = g.neighbors
    labels = {atom: int(label) for atom, label in zip(g.atoms, color_refine_mg(g))}

    label_class_sizes: defaultdict[int, int] = defaultdict(int)
    for label in labels.values():
        label_class_sizes[label] += 1

    labels = {
        atom: (label_class_sizes[label], len(nbrhd[atom]), label)
        for atom, label in labels.items()
    }

    return (
        _Parameters(nbrhds=nbrhd, labels=labels),
        _CanonState(not_included=set(nbrhd)),
    )

In [ ]:
def get_candidates(params: _Parameters, state: _CanonState) -> set[AtomId]:
    """Return the a set of candidate atoms to investigate.
    The candidates are determined based on the current state of the search:
    """

    if not state.frontier:
        # If frontier is empty, start with the smallest label class among not included atoms
        min_label_class_size = min(
            params.labels[atom][0] for atom in state.not_included
        )
        candidates = [
            atom
            for atom in state.not_included
            if params.labels[atom][0] == min_label_class_size
        ]
    else:
        # Otherwise, consider the neighbors of the included atoms
        candidates = []
        for atom in state.frontier:
            for nbr in params.nbrhds[atom]:
                if nbr in state.not_included:
                    candidates.append(nbr)
        # Sort candidates by their labels to ensure deterministic order
        candidates.sort(key=lambda atom: params.labels[atom])

    return set(candidates)

    order = [(v, k) for k, v in params.labels.items()]
    order.sort()

    key, group = next(itertools.groupby(order, key=lambda x: x[0]))

In [ ]:
def canon_atom_num(g: MolGraph) -> Mapping[AtomId, CanonNum]:
    """Return canonical atom order using best-first branching and tie backtracking."""

    params, state = initialize(g)

    state.backtrack_candidates_list.append(get_candidates(params, state))

    atom = state.backtrack_candidates_list[0].pop()

    state.current_labels.append(params.labels[atom])
    state.frontier.update(params.nbrhds[atom])
    # state.current_mapping[atom] = len(state.included)

    while state.backtrack_candidates_list:
        if state.atom_queue:
            next_atom = state.atom_queue.pop()
            state.current_mapping[next_atom] = len(state.current_mapping) + 1

        elif not state.atom_queue:
            backtrack_candidates = state.backtrack_candidates_list[-1]

        if not backtrack_candidates:  # empty list -> backtrack
            ...
        elif backtrack_candidates:
            ...

NameError: name 'MolGraph' is not defined